In [12]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from plot_eval_logs.parse_eval_logs import EvalRun
 

def find_eval_files(root=".", case_insensitive=False, absolute=True):
    """
    Return a sorted list of paths to files with the .eval extension under `root`.

    Args:
        root (str | Path): Directory to search (default: current dir).
        case_insensitive (bool): If True, match extension case-insensitively.
        absolute (bool): If True, return absolute paths; else paths relative to `root`.

    Returns:
        list[pathlib.Path]: Sorted list of paths.
    """
    root = Path(root)
    paths = []
    if case_insensitive:
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() == ".eval":
                paths.append(p.resolve() if absolute else p.relative_to(root))
    else:
        for p in root.rglob("*.eval"):
            if p.is_file():
                paths.append(p.resolve() if absolute else p.relative_to(root))
    return sorted(paths, key=lambda p: str(p))

In [93]:
def parse_into_row(log_path: Path) -> dict:
    log = EvalRun.from_log_file(log_path)
    metrics_dict = {k: round(v.value, 4) for k, v in log.metrics_overall.items()}
    pad_pos = log.task_args["padding_kwargs"].get("padding_position", "before") if log.task_args.get("padding_kwargs", None) else "before"
    row = {
        # "dataset_name": log.task_args["split"],
        "hint_included": log.task_args["hint_included"],
        "detailed_behaviors": log.task_args["detailed_behaviors"],
        # "run_name": str(log_path).replace("/Users/tsimur.hadeliya/code/algoverse_mountain/logs/benign/random_padding", ""),
        "model_name": log.model_name.split("/")[-1],
        "task_type": log.task_name,
        "padding_type": log.task_args["padding_type"],
        "padding_length": log.task_args.get("padding_token_length",log.task_args.get("padding_length",None)),
        "padding_position": pad_pos,
        "random_seed": log.random_seed,
    }
    row.update(metrics_dict)
    return row

In [82]:
logs_benign_paths = find_eval_files(
    "/Users/tsimur.hadeliya/code/algoverse_mountain/logs/benign/random_padding",
    absolute=True
)
logs_harm_paths = find_eval_files(
    "/Users/tsimur.hadeliya/code/algoverse_mountain/logs/harm/random_padding",
    absolute=True
)
logs_paths = logs_benign_paths + logs_harm_paths
rows = []
for log_path in tqdm(logs_paths):
    try:
        row = parse_into_row(log_path)
    except Exception as ex:
        print(f"Error parsing {log_path}: {ex}")
        continue
    rows.append(row)

100%|██████████| 107/107 [02:02<00:00,  1.15s/it]


In [ ]:
df = pd.DataFrame(rows)
df_hier = (
    df.sort_values(["task_type", "padding_type","padding_length", "padding_position","random_seed" ])
      .set_index(["model_name","task_type", "padding_type", "padding_length","padding_position","random_seed"])
)
display(df_hier)

avg_score  \
model_name              task_type padding_type padding_length padding_position random_seed              
gpt-4.1-nano-2025-04-14 benign    random       1000           after            3               0.6207   
                                                                               14              0.5734   
                                                                               15              0.6057   
                                                                               65              0.5986   
                                                                               92              0.6457   
...                                                                                               ...   
                        harmful   random       100000         before           92              0.4768   
                                               100001         after            3               0.3262   
                                                                               3               0.2439   
                                               200000         after            3               0.1838   
                                               800000         after            3               0.1618   

                                                                                            avg_full_score  \
model_name              task_type padding_type padding_length padding_position random_seed                   
gpt-4.1-nano-2025-04-14 benign    random       1000           after            3                    0.2045   
                                                                               14                   0.2045   
                                                                               15                   0.2273   
                                                                               65                   0.2500   
                                                                               92                   0.2955   
...                                                                                                    ...   
                        harmful   random       100000         before           92                   0.1364   
                                               100001         after            3                    0.1364   
                                                                               3                    0.0682   
                                               200000         after            3                    0.0227   
                                               800000         after            3                    0.0227   

                                                                                            avg_refusals  \
model_name              task_type padding_type padding_length padding_position random_seed                 
gpt-4.1-nano-2025-04-14 benign    random       1000           after            3                  0.0000   
                                                                               14                 0.0000   
                                                                               15                 0.0000   
                                                                               65                 0.0000   
                                                                               92                 0.0000   
...                                                                                                  ...   
                        harmful   random       100000         before           92                 0.0682   
                                               100001         after            3                  0.2045   
                                                                               3                  0.1818   
                                               200000         after            3                  0.3864   

In [88]:
df_hier.to_excel("eval_logs_summary.xlsx", sheet_name="test_public")

In [99]:
no_pad_rows = []
for task in ["benign", "harm"]:
    logs_nopad_paths = find_eval_files(
        f"/Users/tsimur.hadeliya/code/algoverse_mountain/logs/{task}/no_padding",
        absolute=True
    )
    for log_path in tqdm(logs_nopad_paths):
        try:
            row = parse_into_row(log_path)
        except Exception as ex:
            print(f"Error parsing {log_path}: {ex}")
            continue
        no_pad_rows.append(row)

100%|██████████| 10/10 [00:03<00:00,  2.76it/s]


In [109]:
df = pd.DataFrame(no_pad_rows)
df = df.fillna("None")               # simplest
# or (keeps control over dtypes)
df = df.astype(object).where(df.notna(), "None")
df_hier = (
    df.sort_values(["model_name", "hint_included","detailed_behaviors", "task_type", "random_seed" ])
      .set_index(["model_name","task_type","hint_included","detailed_behaviors", "random_seed"])
)
df_hier

padding_type  \
model_name              task_type hint_included detailed_behaviors random_seed                
gpt-4.1-nano-2025-04-14 benign    False         False              3                   None   
                                                                   14                  None   
                                                                   15                  None   
                                                                   42                  None   
                                                                   42                  None   
                                                                   42                  None   
                                                                   42                  None   
                                                                   42                  None   
                                                                   65                  None   
                                                                   92                  None   
                                  True          True               3                   None   
                                                                   14                  None   
                                                                   15                  None   
                                                                   42                  None   
                                                                   42                  None   
                                                                   42                  None   
                                                                   42                  None   
                                                                   42                  None   
                                                                   65                  None   
                                                                   92                  None   
                        harmful   True          True               3                   None   
                                                                   3                   None   
                                                                   14                  None   
                                                                   14                  None   
                                                                   15                  None   
                                                                   15                  None   
                                                                   65                  None   
                                                                   65                  None   
                                                                   92                  None   
                                                                   92                  None   
gpt-4o-mini-2024-07-18  benign    True          True               3                   None   
                                                                   14                  None   
                                                                   15                  None   
                                                                   65                  None   
                                                                   92                  None   
gpt-5-nano-2025-08-07   benign    None          None               54636803            None   
                                                                   3133071376          None   
                        harmful   None          None               2680365017          None   

                                                                               padding_length  \
model_name              task_type hint_included detailed_behaviors random_seed                  
gpt-4.1-nano-2025-04-14 benign    False         False              3                

In [110]:
df_hier.to_excel("eval_logs_no_pad.xlsx", sheet_name="test_public")